In [1]:
# %% ==========================================
# 🔥 0. IMPORTS & CONFIG
# ==========================================
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import os
import random
import gc
from tqdm import tqdm

from sklearn.metrics import r2_score, mean_absolute_error
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

# %% ==========================================
# 🔥 1. LOAD DATA
# ==========================================
def load_data(file):
    return pd.read_csv(os.path.join("data", file))

train_df = load_data("train_hotel.csv")
val_df = load_data("valid_hotel.csv")
test_df = load_data("test_hotel.csv")

print(len(train_df), len(val_df), len(test_df))

# %% ==========================================
# 🔥 2. TOKENIZER
# ==========================================
model_id = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

MAX_LEN = 256
STRIDE = 128

def chunk_tokenize(text):
    return tokenizer(
        text,
        truncation=True,
        max_length=MAX_LEN,
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_tensors="pt"
    )

# %% ==========================================
# 🔥 3. CORN UTILS
# ==========================================
def label_to_corn(labels, num_classes=5):
    sets = [(labels > i) for i in range(num_classes - 1)]
    return torch.stack(sets, dim=1).float()

THRESHOLDS = torch.tensor([0.3, 0.5, 0.7, 0.9])

def predict(logits):
    probs = torch.sigmoid(logits)
    return torch.sum(probs > THRESHOLDS.to(probs.device), dim=1) + 1

# %% ==========================================
# 🔥 4. MODEL
# ==========================================
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_id)
        h = self.bert.config.hidden_size

        self.head = nn.Sequential(
            nn.LayerNorm(h),
            nn.Linear(h, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 4)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        mask = attention_mask.unsqueeze(-1)
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

        return self.head(pooled)

# %% ==========================================
# 🔥 5. ERROR CORRECTOR
# ==========================================
class Corrector(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, probs):
        return self.net(probs)

# %% ==========================================
# 🔥 6. INIT
# ==========================================
model = Model().to(device)
corrector = Corrector().to(device)

optimizer = torch.optim.AdamW([
    {'params': model.bert.parameters(), 'lr': 1e-5},
    {'params': model.head.parameters(), 'lr': 1e-4},
    {'params': corrector.parameters(), 'lr': 1e-4}
], weight_decay=0.01)

loss_fn = nn.BCEWithLogitsLoss()

# %% ==========================================
# 🔥 7. TRAIN
# ==========================================
best_loss = 1e9
patience = 2
counter = 0

for epoch in range(10):
    model.train()
    total = 0

    for i in tqdm(range(len(train_df))):
        text = train_df.iloc[i]['Review']
        y = torch.tensor(train_df.iloc[i]['Rating'] - 1).to(device)

        tokens = chunk_tokenize(text)
        input_ids = tokens['input_ids'].to(device)
        mask = tokens['attention_mask'].to(device)

        logits = model(input_ids, mask)
        logits = logits.mean(0)

        target = label_to_corn(y.unsqueeze(0)).to(device)

        loss = loss_fn(logits.unsqueeze(0), target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total += loss.item()

    # VALIDATION
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for i in range(len(val_df)):
            text = val_df.iloc[i]['Review']
            y = torch.tensor(val_df.iloc[i]['Rating'] - 1).to(device)

            tokens = chunk_tokenize(text)
            input_ids = tokens['input_ids'].to(device)
            mask = tokens['attention_mask'].to(device)

            logits = model(input_ids, mask).mean(0)
            target = label_to_corn(y.unsqueeze(0)).to(device)

            val_loss += loss_fn(logits.unsqueeze(0), target).item()

    print(f"Epoch {epoch} | Train {total:.3f} | Val {val_loss:.3f}")

    if val_loss < best_loss:
        best_loss = val_loss
        best_state = model.state_dict()
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping")
            break

model.load_state_dict(best_state)

# %% ==========================================
# 🔥 8. TEST
# ==========================================
model.eval()
corrector.eval()

preds = []
true = test_df['Rating'].values

with torch.no_grad():
    for i in tqdm(range(len(test_df))):
        text = test_df.iloc[i]['Review']

        tokens = chunk_tokenize(text)
        input_ids = tokens['input_ids'].to(device)
        mask = tokens['attention_mask'].to(device)

        logits = model(input_ids, mask).mean(0)
        probs = torch.sigmoid(logits)

        pred = predict(logits.unsqueeze(0)).float()

        # correction
        corr = corrector(probs.unsqueeze(0)).squeeze()
        final = torch.clamp(pred + corr, 1, 5)

        preds.append(final.item())

preds = np.array(preds)

# %% ==========================================
# 🔥 9. METRICS
# ==========================================
r2 = r2_score(true, preds)
mae = mean_absolute_error(true, preds)
acc = (np.round(preds) == true).mean() * 100

print("R2:", r2)
print("MAE:", mae)
print("Accuracy:", acc)

/home/christ/Downloads/Telegram Desktop/M1/M1 Avignon universite/approches neuronales/TP/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


18491 1000 1000


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 598.67it/s, Materializing param=layers.21.mlp_norm.weight]     
ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  0%|          | 0/18491 [00:11<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 148.00 MiB. GPU 0 has a total capacity of 5.64 GiB of which 58.19 MiB is free. Process 348949 has 2.06 GiB memory in use. Process 363258 has 2.06 GiB memory in use. Including non-PyTorch memory, this process has 1.45 GiB memory in use. Of the allocated memory 1.31 GiB is allocated by PyTorch, and 22.70 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)